# ch05 Bonus 16：用真实 LLM 跑 ch05 训练流程

> 对照官方 `ch05/14_ch05_with_other_llms`

## 一句话

把主线 ch05 的训练循环套到 **HuggingFace Transformers 的真实预训练模型**（Llama/Qwen/Gemma）上，验证我们的训练代码对不同架构通用。

## 思路

我们的训练循环（数据加载 → loss → backward → step）是架构无关的。只需把 `model(in_idx)` 换成 HF 模型的 `model(input_ids).logits`，其余完全复用。

> 验证了「训练基础设施」和「模型架构」是解耦的——这是好的工程实践。

In [ ]:
# 演示训练循环对任意 model 对象的通用性
import torch
import torch.nn.functional as F

# 我们的训练循环只依赖：model(x) → logits [b, seq, vocab]
def train_step(model, x, y, optimizer):
    optimizer.zero_grad()
    logits = model(x)              # 不管底层是 GPT/Llama/Qwen，接口一致
    if hasattr(logits, "logits"):  # HF 模型返回对象，取 .logits
        logits = logits.logits
    loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
    loss.backward()
    optimizer.step()
    return loss.item()

# 用我们的 GPT 验证（真实场景换 model = LlamaForCausalLM.from_pretrained(...)）
from src.gpt import GPTModel, GPT_CONFIG_124M
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 64})
torch.manual_seed(123)
model = GPTModel(cfg)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

x = torch.randint(0, cfg["vocab_size"], (2, 16))
y = torch.randint(0, cfg["vocab_size"], (2, 16))

print("用我们的训练循环跑 5 步：")
for step in range(5):
    loss = train_step(model, x, y, opt)
    print(f"  step {step}: loss {loss:.4f}")

print("\n💡 换成 HF 的 LlamaModel 只需：from transformers import LlamaForCausalLM")
print("   model = LlamaForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B')")
print("   其余训练代码一行不用改（model(x).logits 接口一致）。")